<a href="https://colab.research.google.com/github/samrithapalanivel-boop/AI-Government-Scheme-Assistant/blob/main/navigate_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


# ================================================================
# AI-POWERED GOVERNMENT SCHEME ELIGIBILITY & APPLICATION ASSISTANT
# RAG + MULTIMODAL AI + CHROMADB + 4 AI AGENTS + LIVE GOOGLE SEARCH
# GOOGLE COLAB - COMPLETE SINGLE CELL (ERROR-FIXED)
# ================================================================

# 1. INSTALL LIBRARIES
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "google-genai", "chromadb", "sentence-transformers",
    "pymupdf", "pillow"
], check=True)

# 2. IMPORT LIBRARIES
import os
import json
import uuid
import time
import chromadb
import fitz  # PyMuPDF
from getpass import getpass
from PIL import Image
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer
from google.colab import files
from IPython.display import display, Markdown

print("All libraries imported successfully!")

# 3. GEMINI API CONFIGURATION (BASE URL ADDED)
API_KEY = getpass("Enter your Gemini API Key: ").strip()
BASE_URL = "https://generativelanguage.googleapis.com"   # <-- unga base URL inga podunga
client = genai.Client(
    api_key=API_KEY,
    http_options=types.HttpOptions(base_url=BASE_URL)
)
MODEL_NAME = "gemini-2.5-flash"
print("Gemini AI Connected!")


# 4. HELPER FUNCTIONS (SAFE TEXT, SOURCES, GENERATION WITH RETRY)
def get_text(response):
    try:
        return response.text or ""
    except Exception:
        return ""


def get_sources(response):
    """Collect Google Search grounding sources (title - url)."""
    found = []
    try:
        meta = response.candidates[0].grounding_metadata
        for chunk in (meta.grounding_chunks or []):
            if chunk.web and chunk.web.uri:
                found.append(f"{chunk.web.title} - {chunk.web.uri}")
    except Exception:
        pass
    return list(dict.fromkeys(found))


def generate_ai_response(prompt, use_live_search=False, retries=3):
    """Returns (text, sources). Retries on API errors; falls back to no-search."""
    tools = [types.Tool(google_search=types.GoogleSearch())] if use_live_search else None
    config = types.GenerateContentConfig(temperature=0.2, tools=tools)

    last_error = None
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME, contents=prompt, config=config
            )
            text = get_text(response)
            if text.strip():
                return text, get_sources(response)
            last_error = "Empty response"
        except Exception as e:
            last_error = e
        time.sleep(4 * (attempt + 1))

    if use_live_search:
        print("  (Live search failed, retrying without search...)")
        return generate_ai_response(prompt, use_live_search=False, retries=2)

    return f"[AI generation failed: {last_error}]", []


# 5. MULTIMODAL DOCUMENT PROCESSING
def extract_image_information(image_path):
    image = Image.open(image_path).convert("RGB")
    prompt = """
    Analyze this government scheme document image.

    Extract:
    1. Scheme name
    2. Eligibility criteria
    3. Age requirements
    4. Income requirements
    5. Education requirements
    6. Benefits
    7. Required documents
    8. Application procedure
    9. Important dates
    10. Official website or application link

    Read tables and visible text carefully.
    Do not invent missing information.
    Clearly mention information that is unreadable.
    """
    response = client.models.generate_content(
        model=MODEL_NAME, contents=[prompt, image]
    )
    return get_text(response)


print("\nUPLOAD GOVERNMENT SCHEME DOCUMENTS (PDF / PNG / JPG / JPEG / WEBP)")
print("Documents illana 'Cancel upload' click pannalam - Google Search mattum use aagum.")

try:
    uploaded_files = files.upload()
except Exception:
    uploaded_files = {}

document_chunks = []
document_sources = []

for filename in uploaded_files.keys():
    extension = filename.lower().split(".")[-1]
    print("\nProcessing:", filename)
    extracted_text = ""

    try:
        if extension == "pdf":
            pdf = fitz.open(filename)
            for page_number, page in enumerate(pdf):
                page_text = page.get_text().strip()
                if page_text:
                    extracted_text += f"\nPage {page_number + 1}\n{page_text}\n"
                else:
                    # Scanned page -> Gemini Vision OCR
                    pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
                    image_path = f"/tmp/page_{uuid.uuid4().hex}.png"
                    pix.save(image_path)
                    try:
                        extracted_text += f"\nPage {page_number + 1}\n" + \
                            extract_image_information(image_path) + "\n"
                    except Exception as e:
                        print("Scanned page warning:", e)
                    finally:
                        if os.path.exists(image_path):
                            os.remove(image_path)
            pdf.close()

        elif extension in ["png", "jpg", "jpeg", "webp"]:
            extracted_text = extract_image_information(filename)

        else:
            print("Unsupported file skipped:", filename)
            continue

    except Exception as e:
        print("Could not process", filename, "->", e)
        continue

    if extracted_text.strip():
        document_sources.append(filename)
        words = extracted_text.split()
        chunk_size, overlap = 180, 35
        for i in range(0, len(words), chunk_size - overlap):
            chunk = " ".join(words[i:i + chunk_size])
            if len(chunk.strip()) > 30:
                document_chunks.append({"text": chunk, "source": filename})

print("\nDOCUMENT PROCESSING COMPLETED")
print("Documents:", len(document_sources))
print("Total chunks:", len(document_chunks))


# 6. CHROMADB VECTOR DATABASE
print("\nInitializing ChromaDB...")

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("government_scheme_rag")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="government_scheme_rag",
    metadata={"hnsw:space": "cosine"}
)

if document_chunks:
    texts = [c["text"] for c in document_chunks]
    embeddings = embedding_model.encode(
        texts, normalize_embeddings=True, show_progress_bar=True
    ).tolist()
    collection.add(
        ids=[str(i) for i in range(len(texts))],
        documents=texts,
        embeddings=embeddings,
        metadatas=[{"source": c["source"]} for c in document_chunks]
    )

print("ChromaDB Ready! Stored chunks:", collection.count())


# 7. RAG RETRIEVAL ENGINE
def retrieve_context(question, top_k=5):
    if collection.count() == 0:
        return "No uploaded scheme documents available."

    query_embedding = embedding_model.encode(
        [question], normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(top_k, collection.count())
    )

    context = []
    for document, metadata in zip(results["documents"][0], results["metadatas"][0]):
        context.append("SOURCE: " + metadata["source"] + "\nCONTENT: " + document)
    return "\n\n".join(context)


# 8. MULTI-AGENT SYSTEM
print("\nInitializing 4 AI Agents...")


# AGENT 1: PROFILE ANALYSIS
def profile_analysis_agent(user_profile):
    print("\n[AGENT 1] Profile Analysis Agent Running...")
    prompt = f"""
    You are an AI User Profile Analysis Agent.

    USER PROFILE:
    {user_profile}

    Extract and organize:
    1. Applicant age
    2. State
    3. Education
    4. Occupation
    5. Annual family income
    6. Applicant category, if voluntarily provided
    7. Special circumstances, if provided
    8. Missing information

    Never assume missing information.
    Do not infer sensitive personal characteristics.
    Return a structured profile analysis.
    """
    text, _ = generate_ai_response(prompt)
    print("Profile Analysis Completed!")
    return text


# AGENT 2: ELIGIBILITY ANALYSIS
def eligibility_analysis_agent(profile_analysis, retrieved_context, user_question):
    print("\n[AGENT 2] Eligibility Analysis Agent Running...")
    print("Checking current government scheme information...")
    prompt = f"""
    You are an AI Government Scheme Eligibility Agent.

    APPLICANT PROFILE:
    {profile_analysis}

    RETRIEVED RAG DOCUMENTS:
    {retrieved_context}

    USER QUESTION:
    {user_question}

    TASK:
    1. Identify relevant government schemes (Central and the applicant's state).
    2. Check age requirements.
    3. Check income requirements.
    4. Check educational qualifications.
    5. Check state and residency requirements.
    6. Check category or other conditions only if relevant.
    7. Identify missing eligibility information.
    8. Explain why the applicant may or may not qualify.

    Use the uploaded documents and current official information found
    through Google Search. Prefer official websites: .gov.in, .nic.in
    and official ministry or state portals.

    Clearly distinguish:
    - Potentially eligible
    - Not eligible based on known criteria
    - Eligibility cannot yet be determined

    Never claim confirmed eligibility without sufficient evidence.
    Never invent scheme rules or benefits.
    Include source URLs and verification dates where available.
    """
    text, sources = generate_ai_response(prompt, use_live_search=True)
    print("Eligibility Analysis Completed!")
    return text, sources


# AGENT 3: SCHEME COMPARISON
def scheme_comparison_agent(profile_analysis, eligibility_result, retrieved_context):
    print("\n[AGENT 3] Scheme Comparison Agent Running...")
    prompt = f"""
    You are an AI Government Scheme Comparison Agent.

    APPLICANT PROFILE:
    {profile_analysis}

    ELIGIBILITY ANALYSIS:
    {eligibility_result}

    RAG DOCUMENT CONTEXT:
    {retrieved_context}

    Compare the identified government schemes in a table with:
    1. Scheme name
    2. Department or government
    3. Eligibility criteria
    4. Financial or other benefits
    5. Required documents
    6. Application method
    7. Important deadlines
    8. Official application website
    9. Applicant-specific missing requirements

    Compare only schemes supported by available evidence.
    Mark unknown information as "Not verified".
    Do not fabricate financial benefits or deadlines.
    """
    text, sources = generate_ai_response(prompt, use_live_search=True)
    print("Scheme Comparison Completed!")
    return text, sources


# AGENT 4: APPLICATION GUIDANCE
def application_guidance_agent(profile_analysis, eligibility_result,
                               comparison_result, retrieved_context):
    print("\n[AGENT 4] Application Guidance Agent Running...")
    prompt = f"""
    You are an AI Government Scheme Application Guidance Agent.

    PROFILE:
    {profile_analysis}

    ELIGIBILITY:
    {eligibility_result}

    COMPARISON:
    {comparison_result}

    DOCUMENT CONTEXT:
    {retrieved_context}

    Generate a step-by-step application guide including:
    1. Potentially relevant schemes
    2. Required document checklist
    3. Missing information checklist
    4. Official application website
    5. Step-by-step application process
    6. How to track an application
    7. Common application mistakes
    8. Official helpdesk or contact details, if verified

    IMPORTANT:
    - Do not submit applications automatically.
    - Do not request Aadhaar numbers, passwords, OTPs, bank account
      numbers or other sensitive credentials.
    - Never claim that an application was submitted.
    - Tell the user to verify information on official portals.
    - Do not invent application links.
    """
    text, sources = generate_ai_response(prompt, use_live_search=True)
    print("Application Guidance Completed!")
    return text, sources


# 9. MULTI-AGENT ORCHESTRATOR
def multi_agent_app(user_profile, user_question):
    print("\n" + "=" * 65)
    print("STARTING AI MULTI-AGENT WORKFLOW")
    print("=" * 65)

    print("\n[STEP 1] Retrieving scheme documents (RAG)...")
    retrieved_context = retrieve_context(user_question, top_k=5)
    print("RAG Retrieval Completed!")

    profile_result = profile_analysis_agent(user_profile)

    eligibility_result, src2 = eligibility_analysis_agent(
        profile_result, retrieved_context, user_question
    )
    comparison_result, src3 = scheme_comparison_agent(
        profile_result, eligibility_result, retrieved_context
    )
    application_result, src4 = application_guidance_agent(
        profile_result, eligibility_result, comparison_result, retrieved_context
    )

    print("\n[FINAL] Generating Combined AI Report...")
    final_prompt = f"""
    You are the final coordinator of an AI Government Scheme Assistant.
    Combine the outputs from all four agents.

    USER QUESTION:
    {user_question}

    PROFILE AGENT:
    {profile_result}

    ELIGIBILITY AGENT:
    {eligibility_result}

    COMPARISON AGENT:
    {comparison_result}

    APPLICATION GUIDANCE AGENT:
    {application_result}

    Generate a professional final report with these headings:

    # 1. Applicant Profile Summary
    # 2. Identified Government Schemes
    # 3. Eligibility Analysis
    # 4. Scheme Comparison
    # 5. Required Documents
    # 6. Missing Information
    # 7. Step-by-Step Application Guidance
    # 8. Official Websites and Sources
    # 9. Important Verification Notice

    Clearly distinguish verified information from uncertain information.
    Do not create unsupported eligibility decisions.
    Use readable headings and tables.
    """
    final_answer, _ = generate_ai_response(final_prompt, use_live_search=False)

    all_sources = list(dict.fromkeys(src2 + src3 + src4))
    if all_sources:
        final_answer += "\n\n---\n## Live Google Search Sources\n" + \
            "\n".join(f"- {s}" for s in all_sources)

    return {
        "profile_analysis": profile_result,
        "eligibility_result": eligibility_result,
        "comparison_result": comparison_result,
        "application_guidance": application_result,
        "final_answer": final_answer,
        "retrieved_context": retrieved_context,
    }


# 10. USER INPUT (ASKED ONLY ONCE)
print("\n" + "=" * 65)
print("ENTER APPLICANT DETAILS")
print("(Aadhaar, bank details, OTP, password ethuvum enter pannadheenga)")
print("=" * 65)

user_profile = {
    "Age": input("Age: ").strip(),
    "State": input("State: ").strip(),
    "Education": input("Education: ").strip(),
    "Occupation": input("Occupation: ").strip(),
    "Annual Family Income (INR)": input("Annual Family Income (INR): ").strip(),
    "Applicant Category": input("Applicant Category (optional): ").strip(),
    "Additional Information": input("Additional Information (optional): ").strip(),
}

profile_text = json.dumps(user_profile, indent=2, ensure_ascii=False)

user_question = input("\nEnter your government scheme question: ").strip()
if not user_question:
    user_question = (
        "Find relevant government schemes, analyze eligibility, "
        "compare benefits and explain application procedures."
    )


# 11. RUN WORKFLOW
try:
    result = multi_agent_app(user_profile=profile_text, user_question=user_question)
except Exception as e:
    raise SystemExit(f"Workflow error: {type(e).__name__}: {e}")


# 12. DISPLAY FINAL REPORT
print("\n" + "=" * 65)
print("AI GOVERNMENT SCHEME ASSISTANT - FINAL REPORT")
print("=" * 65)
display(Markdown(result["final_answer"]))


# 13. INDIVIDUAL AGENT OUTPUTS
print("\n" + "=" * 65)
print("INDIVIDUAL AGENT REPORTS")
print("=" * 65)

agent_reports = {
    "Agent 1 - Profile Analysis": result["profile_analysis"],
    "Agent 2 - Eligibility Analysis": result["eligibility_result"],
    "Agent 3 - Scheme Comparison": result["comparison_result"],
    "Agent 4 - Application Guidance": result["application_guidance"],
}

for agent_name, agent_output in agent_reports.items():
    display(Markdown(f"## {agent_name}"))
    display(Markdown(str(agent_output)))


# 14. SAVE & DOWNLOAD REPORT
report_filename = "Government_Scheme_AI_Report.txt"

with open(report_filename, "w", encoding="utf-8") as f:
    f.write("AI GOVERNMENT SCHEME ASSISTANT\n\n")
    f.write(result["final_answer"])
    f.write("\n\n--- INDIVIDUAL AGENT REPORTS ---\n")
    for name, output in agent_reports.items():
        f.write(f"\n\n{name}\n")
        f.write(str(output))

files.download(report_filename)

print("\n" + "=" * 65)
print("PROJECT EXECUTION COMPLETED!")
print("RAG: READY | MULTIMODAL: READY | CHROMADB: READY")
print("4 AGENTS: COMPLETED | FINAL REPORT: GENERATED")
print("=" * 65)

All libraries imported successfully!
Enter your Gemini API Key: ··········
Gemini AI Connected!

UPLOAD GOVERNMENT SCHEME DOCUMENTS (PDF / PNG / JPG / JPEG / WEBP)
Documents illana 'Cancel upload' click pannalam - Google Search mattum use aagum.


Saving it_e_pn_2023_24.pdf to it_e_pn_2023_24 (2).pdf

Processing: it_e_pn_2023_24 (2).pdf

DOCUMENT PROCESSING COMPLETED
Documents: 1
Total chunks: 114

Initializing ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

ChromaDB Ready! Stored chunks: 114

Initializing 4 AI Agents...

ENTER APPLICANT DETAILS
(Aadhaar, bank details, OTP, password ethuvum enter pannadheenga)
Age: 21
State: TAMILNADU
Education: BE 4th year
Occupation: STUDENT
Annual Family Income (INR): 100000
Applicant Category (optional): STUDENT
Additional Information (optional): I am a final-year Engineering student 

Enter your government scheme question: Which government scholarships are available for undergraduate students in Tamil Nadu?

STARTING AI MULTI-AGENT WORKFLOW

[STEP 1] Retrieving scheme documents (RAG)...
RAG Retrieval Completed!

[AGENT 1] Profile Analysis Agent Running...
Profile Analysis Completed!

[AGENT 2] Eligibility Analysis Agent Running...
Checking current government scheme information...
  (Live search failed, retrying without search...)
Eligibility Analysis Completed!

[AGENT 3] Scheme Comparison Agent Running...
  (Live search failed, retrying without search...)
Scheme Comparison Completed!

[AGENT 4] Appli

[AI generation failed: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}]

... (8 outputs left out)

[AI generation failed: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


PROJECT EXECUTION COMPLETED!
RAG: READY | MULTIMODAL: READY | CHROMADB: READY
4 AGENTS: COMPLETED | FINAL REPORT: GENERATED
